In [ ]:
%run ./catalog_slug

In [ ]:
# =============================================================================
# Training environment setup: run with %run ../../setup/00_setup from every module and lab.
# It finds your catalog retailhub_<you> (created by the trainer with 00_pre_config), checks the
# bronze/silver/gold/default schemas and the dataset Volume, and exports CATALOG, BRONZE_SCHEMA,
# SILVER_SCHEMA, GOLD_SCHEMA, DEFAULT_SCHEMA and DATASET_PATH. On success it prints one line.
# Do not modify this notebook.
# =============================================================================
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"
DEFAULT_SCHEMA = "default"
VOLUME_NAME = "datasets"

# 1. Your login -> catalog name (same rule as 00_pre_config, see catalog_slug)
raw_user = spark.sql("SELECT current_user()").first()[0]
user_slug = catalog_slug(raw_user)
CATALOG = f"{CATALOG_PREFIX}_{user_slug}"

# 2. The catalog must exist (the trainer creates it before the training)
catalogs = [row[0] for row in spark.sql("SHOW CATALOGS").collect()]
if CATALOG not in catalogs:
    course = [c for c in catalogs if c.startswith(CATALOG_PREFIX)]
    raise Exception(f"Catalog '{CATALOG}' not found for {raw_user}. Course catalogs you can see: {course}. "
                    "Ask the trainer to run 00_pre_config.")
spark.sql(f"USE CATALOG {CATALOG}")

# 3. The schemas must exist
schema_names = [row[0] for row in spark.sql(f"SHOW SCHEMAS IN {CATALOG}").collect()]
missing = [s for s in (BRONZE_SCHEMA, SILVER_SCHEMA, GOLD_SCHEMA, DEFAULT_SCHEMA) if s not in schema_names]
if missing:
    raise Exception(f"Missing schemas in {CATALOG}: {missing}. Ask the trainer.")

# 4. The dataset Volume must be readable (raises with the reason if not)
DATASET_PATH = f"/Volumes/{CATALOG}/{DEFAULT_SCHEMA}/{VOLUME_NAME}"
dbutils.fs.ls(DATASET_PATH)

# Aliases kept for older notebooks
catalog = CATALOG
volume_path = DATASET_PATH

print(f"Your catalog: {CATALOG} | schemas: {BRONZE_SCHEMA}, {SILVER_SCHEMA}, {GOLD_SCHEMA} | dataset: {DATASET_PATH}")